# Lab: escolhendo o melhor modelo com validação cruzada

Notebook de aula da página: https://insper.github.io/ML/pt/2026.2/classes/validation/

Rode as células em ordem. Em cada passo, responda o **Preveja** com a dupla *antes* de rodar. Nas **Simulações**, escreva o experimento na célula vazia; as soluções e a leitura de cada resultado estão na página da aula.

## Passo 1 — carregar e corrigir o rótulo

**Preveja.** No conjunto do scikit-learn, `target = 1` é benigno. Se você calcular o "recall" sem mexer nisso, o que exatamente estaria medindo?

In [1]:
import numpy as np
import pandas as pd
from sklearn.datasets import load_breast_cancer

data = load_breast_cancer(as_frame=True)
X = data.data                     # 30 medidas numéricas de cada tumor
y = 1 - data.target               # o scikit-learn codifica benigno = 1; invertemos para 1 = maligno
print(X.shape)
print(y.value_counts())

(569, 30)
target
0    357
1    212
Name: count, dtype: int64


**Resposta.** Estaria medindo o recall da classe benigna, ou seja, a fração de tumores benignos reconhecidos como benignos, e não a fração de malignos detectados, que é o que interessa. Depois da inversão, `y = 1` marca os 212 malignos e `y = 0` os 357 benignos.

## Passo 2 — trancar o conjunto de teste

**Preveja.** Esta é a única divisão feita à mão no lab inteiro. Por que o `stratify` importa aqui, com 37% de malignos? Até que passo o `X_test` não pode ser tocado?

In [2]:
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, stratify=y, random_state=42)
print(X_train.shape, X_test.shape)
print(f"fração maligna: treino {y_train.mean():.3f} · teste {y_test.mean():.3f}")

(455, 30) (114, 30)
fração maligna: treino 0.374 · teste 0.368


**Resposta.** Com 37% de malignos e só 114 linhas de teste, uma divisão sem `stratify` poderia deixar alguns pontos a mais ou a menos de malignos de um lado, mudando o denominador do recall; estratificado, o treino fica com 0,374 e o teste com 0,368. O `X_test` não pode ser tocado até o passo 12, o único teste.

## Passo 3 — um baseline, para os números significarem algo

**Preveja.** Um modelo que sempre responde "benigno" vai tirar que acurácia, com 37% de malignos? E que recall?

In [3]:
from sklearn.dummy import DummyClassifier
from sklearn.model_selection import StratifiedKFold, cross_val_score

cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

dummy = DummyClassifier(strategy='most_frequent')          # sempre diz "benigno"
for metric in ['accuracy', 'recall']:
    s = cross_val_score(dummy, X_train, y_train, cv=cv, scoring=metric)
    print(f"{metric:8s} {s.mean():.3f}")

accuracy 0.626
recall   0.000


**Resposta.** A acurácia é a fração de benignos no treino, 0,626, e o recall é 0,000, porque o modelo nunca diz "maligno". É o patamar mínimo: 63% de acurácia sem detectar um único tumor.

## Passo 4 — CV 5-fold com várias métricas

**Preveja.** Os cinco folds vão dar o mesmo recall? Se variarem, você diria que a diferença vem do modelo ou de quais tumores caíram em cada fold?

In [4]:
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import cross_validate
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler

model = make_pipeline(StandardScaler(), LogisticRegression(max_iter=1000))

res = cross_validate(model, X_train, y_train, cv=cv, return_train_score=True,
                     scoring=['accuracy', 'precision', 'recall', 'f1', 'roc_auc'])
folds = pd.DataFrame(res).drop(columns=['fit_time', 'score_time'])
print(folds[['train_recall', 'test_accuracy', 'test_precision', 'test_recall', 'test_roc_auc']].round(3))
print(folds.agg(['mean', 'std']).T.round(3))

   train_recall  test_accuracy  test_precision  test_recall  test_roc_auc
0         0.978          0.956           1.000        0.882         0.995
1         0.963          0.967           0.943        0.971         0.999
2         0.963          1.000           1.000        1.000         1.000
3         0.978          0.978           1.000        0.941         0.987
4         0.971          0.967           0.943        0.971         0.998


                  mean    std
test_accuracy    0.974  0.017
train_accuracy   0.988  0.004
test_precision   0.977  0.031
train_precision  0.997  0.004
test_recall      0.953  0.045
train_recall     0.971  0.007
test_f1          0.964  0.023
train_f1         0.984  0.005
test_roc_auc     0.996  0.005
train_roc_auc    0.998  0.001


**Resposta.** Não: o recall vai de 0,882 a 1,000 entre os folds (0,953 ± 0,045), com a mesma receita em todos. A diferença vem de quais tumores caíram em cada fold: com 34 malignos por fold de validação, um tumor vale cerca de 2,9 pontos de recall, e o fold 0 perdeu 4 deles.

**Simulação: a sorte da divisão, medida.** A aula afirmou que um único split é uma loteria. Meça isso nos dados do lab: para 20 sementes, compare a acurácia de um único hold-out 80/20 com a média de uma CV 5-fold. Que faixa cada um percorre?

In [5]:
from sklearn.base import clone

holdout, cv_means = [], []
for seed in range(20):
    X_tr, X_va, y_tr, y_va = train_test_split(X_train, y_train, test_size=0.2,
                                              stratify=y_train, random_state=seed)
    holdout.append(clone(model).fit(X_tr, y_tr).score(X_va, y_va))
    cv_seed = StratifiedKFold(n_splits=5, shuffle=True, random_state=seed)
    cv_means.append(cross_val_score(model, X_train, y_train, cv=cv_seed).mean())
for name, s in [('hold-out 80/20', np.array(holdout)), ('média da CV 5-fold', np.array(cv_means))]:
    print(f"{name:19s} de {s.min():.3f} a {s.max():.3f} · desvio {s.std():.3f}")

hold-out 80/20      de 0.934 a 1.000 · desvio 0.016
média da CV 5-fold  de 0.969 a 0.980 · desvio 0.003


**Leitura.** Em 20 sementes, o hold-out 80/20 vai de 0,934 a 1,000 (desvio 0,016), enquanto a média da CV 5-fold fica entre 0,969 e 0,980 (desvio 0,003). Ou seja, um único split oscila cerca de 5 vezes mais e, dependendo da sorte, reporta um modelo perfeito ou 6,6 pontos pior. A média de cinco folds é o número que se deve reportar.

## Passo 5 — uma média mais estável: CV repetida

**Preveja.** Repetir a CV 5-fold dez vezes com embaralhamentos diferentes muda mais a média ou a confiança que você tem nela?

In [6]:
from sklearn.model_selection import RepeatedStratifiedKFold

rcv = RepeatedStratifiedKFold(n_splits=5, n_repeats=10, random_state=42)
rec = cross_val_score(model, X_train, y_train, cv=rcv, scoring='recall')
print(f"{len(rec)} escores · recall {rec.mean():.3f} ± {rec.std():.3f} · "
      f"de {rec.min():.3f} a {rec.max():.3f}")

50 escores · recall 0.947 ± 0.037 · de 0.853 a 1.000


**Resposta.** Muda mais a confiança: a média de recall vai a 0,947, perto do 0,953 da CV única, mas agora vem de 50 escores (desvio 0,037, de 0,853 a 1,000), então ela depende bem menos de um embaralhamento específico. O desvio por fold não diminui, porque cada fold continua do mesmo tamanho.

## Passo 6 — ajustar `C` com uma busca em grade

**Preveja.** `C` pequeno é penalidade forte. Com `C = 0,001`, o recall de treino e o de validação vão ficar os dois baixos, os dois altos ou longe um do outro? E com `C = 1000`?

Lembre que o `C` faz o papel de $1/\alpha$, o inverso da força da penalidade $\alpha$ do [Ridge](https://insper.github.io/ML/pt/2026.2/classes/gradient-descent-regularization/#regularizacao) (a menos de um fator constante): `C` pequeno = penalidade forte.

In [7]:
from sklearn.model_selection import GridSearchCV

grid = GridSearchCV(model, {'logisticregression__C': np.logspace(-3, 3, 13)},
                    cv=cv, scoring='recall', return_train_score=True)
grid.fit(X_train, y_train)

table = pd.DataFrame(grid.cv_results_)[['param_logisticregression__C', 'mean_train_score',
                                        'mean_test_score', 'std_test_score']]
table.columns = ['C', 'recall treino', 'recall val', 'desvio val']
print(table.round(3).to_string(index=False))
print(f"melhor: C = {grid.best_params_['logisticregression__C']:g} · recall {grid.best_score_:.3f}")

       C  recall treino  recall val  desvio val
   0.001          0.694       0.700       0.088
   0.003          0.843       0.841       0.084
   0.010          0.878       0.865       0.061
   0.032          0.929       0.912       0.042
   0.100          0.957       0.935       0.043
   0.316          0.965       0.947       0.051
   1.000          0.971       0.953       0.040
   3.162          0.978       0.941       0.037
  10.000          0.978       0.941       0.032
  31.623          0.985       0.941       0.032
 100.000          0.997       0.947       0.022
 316.228          1.000       0.947       0.022
1000.000          1.000       0.947       0.022
melhor: C = 1 · recall 0.953


**Resposta.** Com `C = 0,001`, os dois ficam baixos e juntos (treino 0,694, validação 0,700): é subajuste. Com `C = 1000`, ficam longe um do outro (treino 1,000, validação 0,947): o modelo decora o treino, embora a validação não despenque. O melhor é `C = 1`, com recall 0,953.

**Simulação: a regra de um erro-padrão.** A busca escolheu `C = 1`, mas a figura sugere um platô. Aplique a **regra de um erro-padrão** (Breiman et al., 1984): calcule o erro-padrão da melhor média (desvio entre folds dividido por $\sqrt{5}$), veja quantas configurações caem dentro dele e fique com a mais simples — aqui, o `C` mais regularizado.

In [8]:
cvr = pd.DataFrame(grid.cv_results_)
se = cvr.loc[grid.best_index_, 'std_test_score'] / np.sqrt(5)      # erro-padrão da melhor média
within = cvr[cvr['mean_test_score'] >= grid.best_score_ - se]
simplest = within.loc[within['param_logisticregression__C'].idxmin()]
print(f"melhor: C = {grid.best_params_['logisticregression__C']:g} · recall {grid.best_score_:.3f} · "
      f"erro-padrão {se:.3f}")
print(f"{len(within)} de {len(cvr)} configurações dentro de 1 erro-padrão: "
      f"C de {within['param_logisticregression__C'].min():.3g} a {within['param_logisticregression__C'].max():.3g}")
print(f"a mais simples: C = {simplest['param_logisticregression__C']:.3g} · "
      f"recall {simplest['mean_test_score']:.3f}")

melhor: C = 1 · recall 0.953 · erro-padrão 0.018
9 de 13 configurações dentro de 1 erro-padrão: C de 0.1 a 1e+03
a mais simples: C = 0.1 · recall 0.935


**Leitura.** O erro-padrão da melhor média é 0,040/√5 ≈ 0,018, e 9 das 13 configurações ficam a menos dele do recall 0,953, com `C` de 0,1 a 1000. A regra de um erro-padrão escolhe a mais regularizada, `C = 0,1` (recall 0,935, bem na borda). O platô diz que os dados não distinguem `C = 1` de vizinhos como 0,1 ou 10; a busca escolheu `C = 1` por uma diferença menor que o ruído.

**Para fechar a parte fundamental.** Em uma frase cada: (1) o que a validação cruzada avalia, um modelo ou uma receita? (2) Por que o desvio entre folds precisa ser reportado junto com a média? (3) O que acontece se o `StandardScaler` for ajustado antes do `cross_val_score`?

**Resposta.**
1. Uma receita: o pipeline com seus hiperparâmetros e o procedimento de ajuste, reajustado do zero em cada fold, e não um modelo já treinado.
2. Porque a variação entre folds (o recall foi de 0,882 a 1,000) é do mesmo tamanho das diferenças entre modelos, e uma média sozinha não diz se uma diferença é sinal ou ruído.
3. O scaler aprende média e desvio também das linhas de validação, um vazamento que torna a estimativa otimista (aqui pequeno, mas errado em princípio).

## Sua vez (fundamental)

Para casa:

1. **Quantos folds?** Repita o passo 4 com `cv=StratifiedKFold(3, ...)` e com `10`. O que acontece com a média e com o desvio entre folds? Quanto custa em tempo de ajuste?
2. **Sem estratificar.** Troque o `StratifiedKFold` do passo 3 por `KFold(5, shuffle=True, random_state=42)` e olhe quantos malignos caem em cada fold de validação (`for _, va in cv.split(X_train, y_train): print(y_train.iloc[va].sum())`). Com 37% de positivos a diferença é pequena; refaça imaginando 3% e explique por que a estratificação viraria obrigatória.
3. **Fora do pipeline.** Ajuste um `StandardScaler` em todo o `X_train`, transforme e rode o `cross_val_score` só com a `LogisticRegression`. Compare com o passo 4. A diferença é grande aqui? Por que você não pode concluir daí que escalonar fora do pipeline é seguro?

## Passo 7 — ajustar os candidatos nos mesmos folds

**Preveja.** Cada família ganha a própria grade. Por que as duas precisam usar o **mesmo** objeto `cv`? O que você mediria se ajustasse a logística e deixasse o k-NN no padrão?

In [9]:
from sklearn.neighbors import KNeighborsClassifier

knn = make_pipeline(StandardScaler(), KNeighborsClassifier())
search_lr = GridSearchCV(model, {'logisticregression__C': np.logspace(-3, 3, 13),
                                 'logisticregression__class_weight': [None, 'balanced']},
                         cv=cv, scoring=['recall', 'roc_auc'], refit='recall')
search_knn = GridSearchCV(knn, {'kneighborsclassifier__n_neighbors': [1, 3, 5, 7, 9, 11, 15, 21, 31],
                                'kneighborsclassifier__weights': ['uniform', 'distance']},
                          cv=cv, scoring=['recall', 'roc_auc'], refit='recall')

for name, search in [('logística', search_lr), ('k-NN', search_knn)]:
    search.fit(X_train, y_train)
    i = search.best_index_
    best_params = ', '.join(f"{k.split('__')[1]}={v:.3g}" if isinstance(v, float) else f"{k.split('__')[1]}={v}"
                            for k, v in search.best_params_.items())
    print(f"{name:9s} {len(search.cv_results_['params'])} configurações · melhor: {best_params}")
    print(f"{'':9s} recall {search.best_score_:.3f} · ROC AUC {search.cv_results_['mean_test_roc_auc'][i]:.3f}")

logística 26 configurações · melhor: C=0.316, class_weight=balanced
          recall 0.959 · ROC AUC 0.995


k-NN      18 configurações · melhor: n_neighbors=3, weights=uniform
          recall 0.935 · ROC AUC 0.989


**Resposta.** Com o mesmo `cv`, as duas famílias são medidas nas mesmas linhas e a diferença entre elas vem do modelo, não da sorte dos folds, o que permite comparar fold a fold. Ajustando só a logística e deixando o k-NN no padrão, você mediria o ganho do ajuste, não qual família é melhor. Aqui a logística ajustada (`C = 0,316`, `class_weight='balanced'`) faz recall 0,959 e o k-NN ajustado ($k = 3$) 0,935.

## Passo 8 — comparar tudo nos mesmos folds

**Preveja.** A logística ajustada está à frente do k-NN ajustado por cerca de 0,024 de recall e os desvios entre folds são 0,030 e 0,039. Isso basta para declarar um vencedor?

In [10]:
candidates = {'dummy': dummy,
              'logistic, default': model,
              'logistic, tuned': search_lr.best_estimator_,
              'k-NN, default': knn,
              'k-NN, tuned': search_knn.best_estimator_}
scores = {name: cross_validate(est, X_train, y_train, cv=cv, scoring=['recall', 'roc_auc'])
          for name, est in candidates.items()}
summary = pd.DataFrame({name: {'recall': s['test_recall'].mean(), 'recall std': s['test_recall'].std(),
                               'ROC AUC': s['test_roc_auc'].mean(), 'AUC std': s['test_roc_auc'].std()}
                        for name, s in scores.items()}).T
print(summary.round(3))

                   recall  recall std  ROC AUC  AUC std
dummy               0.000       0.000    0.500    0.000
logistic, default   0.953       0.040    0.996    0.005
logistic, tuned     0.959       0.030    0.995    0.005
k-NN, default       0.918       0.034    0.987    0.014
k-NN, tuned         0.935       0.039    0.989    0.013


**Resposta.** Não basta: a vantagem de 0,024 (0,959 contra 0,935) é menor que qualquer um dos dois desvios (0,030 e 0,039), e na ROC AUC a distância é de 0,006 (0,995 contra 0,989). Os dois ajustados batem os padrões e o dummy, mas entre eles a tabela não declara vencedor.

## Passo 9 — diferenças pareadas e o vencedor

**Preveja.** Em vez de comparar duas médias, compare fold a fold. Você espera que a logística vença em todos os cinco folds ou que o sinal da diferença troque?

In [11]:
a, b = scores['logistic, tuned'], scores['k-NN, tuned']
d_recall = a['test_recall'] - b['test_recall']
d_auc = a['test_roc_auc'] - b['test_roc_auc']
print(pd.DataFrame({'recall logística': a['test_recall'], 'recall k-NN': b['test_recall'],
                    'tumores': np.round(d_recall * 34).astype(int),   # 34 malignos por fold de validação
                    'diferença AUC': d_auc}).round(3))
for name, d in [('recall', d_recall), ('ROC AUC', d_auc)]:
    print(f"{name:7s} diferença: média {d.mean():+.3f} · desvio {d.std():.3f} · "
          f"logística à frente em {(d > 0).sum()} de 5 folds")

   recall logística  recall k-NN  tumores  diferença AUC
0             0.912        0.941       -1         -0.003
1             0.971        1.000       -1         -0.001
2             1.000        0.912        3          0.003
3             0.941        0.882        2          0.020
4             0.971        0.941        1          0.013
recall  diferença: média +0.024 · desvio 0.047 · logística à frente em 3 de 5 folds
ROC AUC diferença: média +0.006 · desvio 0.009 · logística à frente em 3 de 5 folds


**Resposta.** O sinal troca: a logística vence em 3 dos 5 folds de recall e perde em 2 (−1, −1, +3, +2 e +1 tumor). A diferença média é +0,024 com desvio 0,047, maior que a própria média, então o resultado é um empate; na ROC AUC o padrão se repete (+0,006 ± 0,009, 3 de 5).

**Simulação: mais folds mudam o veredito?** Cinco folds é pouco. Refaça a comparação pareada com `RepeatedStratifiedKFold(5, n_repeats=10)`, que dá 50 pares, e conte em quantos cada modelo ficou à frente. O empate do passo 9 se mantém?

In [12]:
pair = {name: cross_val_score(est, X_train, y_train, cv=rcv, scoring='recall')   # rcv do passo 5: 5 x 10
        for name, est in [('logística', search_lr.best_estimator_), ('k-NN', search_knn.best_estimator_)]}
for name, s in pair.items():
    print(f"{name:9s} recall {s.mean():.3f} ± {s.std():.3f}")
d = pair['logística'] - pair['k-NN']
tied = np.isclose(d, 0)
print(f"diferença: média {d.mean():+.3f} · desvio {d.std():.3f}")
print(f"logística à frente em {((d > 0) & ~tied).sum()}, empate em {tied.sum()}, "
      f"atrás em {((d < 0) & ~tied).sum()} de {len(d)} folds")

logística recall 0.957 ± 0.032
k-NN      recall 0.928 ± 0.048
diferença: média +0.029 · desvio 0.041
logística à frente em 29, empate em 15, atrás em 6 de 50 folds


**Leitura.** Com 50 pares, a logística faz 0,957 ± 0,032 e o k-NN 0,928 ± 0,048; a diferença é +0,029 ± 0,041, ainda menor que a própria dispersão, então pela regra do passo 9 continua um empate. Mas a contagem favorece a logística: ela fica à frente em 29 folds, empata em 15 e perde em só 6. Não há motivo para preferir o k-NN, e a logística é mais simples.

## Passo 10 — teste de sanidade: embaralhe os rótulos

**Preveja.** Com os rótulos embaralhados, não há nada a aprender. Você espera que o recall caia para zero, para 0,5 ou para algum outro valor? (Dica: o vencedor usa `class_weight='balanced'`.)

In [13]:
from sklearn.model_selection import permutation_test_score

best = search_lr.best_estimator_                 # o vencedor do passo 9
score, perm_scores, p_value = permutation_test_score(
    best, X_train, y_train, cv=cv, scoring='recall', n_permutations=100, random_state=0)
print(f"rótulos reais: {score:.3f}")
print(f"rótulos embaralhados: média {perm_scores.mean():.3f}, melhor de 100 {perm_scores.max():.3f}")
print(f"p-valor: {p_value:.4f}")

rótulos reais: 0.959
rótulos embaralhados: média 0.480, melhor de 100 0.582
p-valor: 0.0099


**Resposta.** Nem zero nem 0,5 exato: com `class_weight='balanced'` o modelo sem sinal chuta "maligno" mais ou menos metade das vezes, então o recall fica perto de 0,5. Os 100 embaralhamentos deram média 0,480 e no máximo 0,582, contra 0,959 com os rótulos reais; o p-valor de 0,0099 = 1/101 é o menor possível com 100 permutações.

## Passo 11 — previsões fora do fold

**Preveja.** Por que uma matriz de confusão feita com `cross_val_predict` é honesta, enquanto uma feita com `best.predict(X_train)` não é?

In [14]:
from sklearn.metrics import confusion_matrix
from sklearn.model_selection import cross_val_predict

oof = cross_val_predict(best, X_train, y_train, cv=cv)      # cada linha prevista por um modelo que nunca a viu
print(confusion_matrix(y_train, oof, labels=[1, 0]))

[[163   7]
 [  4 281]]


**Resposta.** No `cross_val_predict` cada linha é prevista por um modelo que nunca a viu, então os erros são os que o modelo cometeria em dados novos; com `best.predict(X_train)` o modelo julga linhas em que treinou, e os erros saem subestimados. Fora do fold, o vencedor acerta 163 dos 170 malignos (recall 0,959) e dá 4 falsos alarmes em 285 benignos.

## Passo 12 — o único teste

**Preveja.** A CV disse recall 0,959 ± 0,030. O teste vai cair dentro dessa faixa? Com 42 malignos no teste, quanto vale um tumor em pontos de recall?

In [15]:
from sklearn.metrics import accuracy_score, recall_score, roc_auc_score

pred = best.predict(X_test)                 # o GridSearchCV já reajustou o vencedor em todo o X_train
proba = best.predict_proba(X_test)[:, 1]
print(f"teste: acurácia {accuracy_score(y_test, pred):.3f} · recall {recall_score(y_test, pred):.3f}"
      f" · ROC AUC {roc_auc_score(y_test, proba):.3f}")
print(confusion_matrix(y_test, pred, labels=[1, 0]))

teste: acurácia 0.982 · recall 0.976 · ROC AUC 0.997
[[41  1]
 [ 1 71]]


**Resposta.** Caiu dentro da faixa, até acima da média: recall 0,976 no teste (41 dos 42 malignos), contra 0,959 ± 0,030 na CV, com acurácia 0,982 e ROC AUC 0,997. Um tumor vale 1/42 ≈ 2,4 pontos de recall, então a diferença para a CV é menos de um tumor.

**Simulação: quanto o próprio ajuste infla o número?** O `best_score_` de 0,959 é o melhor de 26 configurações, medidas nos mesmos folds que o escolheram. Avalie **a busca da logística** (as 26 configurações) com validação cruzada aninhada — `cross_val_score(search_lr, ...)` num `cv` externo — e compare os três números: busca, CV aninhada e teste.

In [16]:
outer = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
nested = cross_val_score(search_lr, X_train, y_train, cv=outer, scoring='recall')   # cada fold refaz as 26 configs
print(f"busca (best_score_): {search_lr.best_score_:.3f}")
print(f"CV aninhada:         {nested.mean():.3f} ± {nested.std():.3f} · folds {nested.round(3)}")
print(f"teste (passo 12):    {recall_score(y_test, pred):.3f}")

busca (best_score_): 0.959
CV aninhada:         0.947 ± 0.022 · folds [0.912 0.971 0.941 0.941 0.971]
teste (passo 12):    0.976


**Leitura.** A busca diz 0,959, a CV aninhada 0,947 ± 0,022 e o teste 0,976. A CV aninhada fica 1,2 ponto abaixo do `best_score_`, na direção da maldição do vencedor, mas isso é menos de meio tumor por fold (um tumor vale 2,9 pontos), dentro do ruído. Com 26 configurações num platô largo, o próprio ajuste quase não infla o número; o teste acima dos dois também é ruído de 42 tumores.

## Passo 13 — o que o modelo aprendeu?

**Preveja.** Os atributos estão padronizados, então os coeficientes são comparáveis. Você espera que os maiores sejam medidas de tamanho, de textura ou de forma? E eles seriam estáveis se o modelo fosse reajustado em outro fold?

In [17]:
coef = pd.Series(best[-1].coef_[0], index=X.columns)
print(coef.sort_values(key=abs, ascending=False).head(6).round(2))

worst texture          0.96
radius error           0.80
worst symmetry         0.72
worst radius           0.68
mean concave points    0.64
worst area             0.64
dtype: float64


**Resposta.** Uma mistura: o maior é de textura (`worst texture`, 0,96), seguido de tamanho (`radius error` 0,80, `worst radius` 0,68, `worst area` 0,64) e forma (`worst symmetry` 0,72, `mean concave points` 0,64). Provavelmente não seriam estáveis: as medidas de tamanho (raio, perímetro, área) são quase colineares, então o peso se reparte entre elas de um jeito arbitrário que muda de fold para fold.

**Para fechar a aula.** Em uma frase cada: (1) qual dos cinco vazamentos nenhum esquema de CV consegue pegar? (2) Por que comparar fold a fold enxerga diferenças que duas médias ± desvio escondem? (3) Por que o `best_score_` de uma busca em grade é otimista?

**Resposta.**
1. Um atributo que "conhece o futuro" (registrado depois do momento da previsão): ele está presente em todos os folds, então nenhum esquema de divisão o detecta, só o conhecimento do problema.
2. Porque os modelos são medidos nas mesmas linhas, e a diferença fold a fold mostra o sinal e a dispersão da própria diferença, enquanto duas faixas de média ± desvio se sobrepõem mesmo quando um modelo vence de forma consistente (29 contra 6 nos 50 pares).
3. Porque é o máximo de muitas configurações medidas nos mesmos folds que a escolheram, e o vencedor tende a ser também o que teve mais sorte nesses folds (aqui 0,959 contra 0,947 da CV aninhada).

## Sua vez (avançado)

Para casa, com mais tempo:

1. **Limiar.** Use `cross_val_predict(best, X_train, y_train, cv=cv, method='predict_proba')[:, 1]` para obter probabilidades fora do fold. Encontre o maior limiar que dá recall ≥ 0,98 e reporte a precisão que você paga por ele. Depois aplique esse limiar, uma vez, ao conjunto de teste.
2. **Penalidade L1.** Troque por `LogisticRegression(l1_ratio=1, solver='liblinear')` (no scikit-learn anterior à 1.8, escreva `penalty='l1'`) e repita o passo 6. Quantos dos 30 coeficientes são exatamente zero no melhor `C`? O recall muda mais de um desvio-padrão?
3. **A escolha inteira, aninhada.** A simulação do passo 12 aninhou só a busca da logística. Escreva o laço externo você mesmo, como na [seção de CV aninhada](https://insper.github.io/ML/pt/2026.2/classes/validation/#a-maldicao-do-vencedor-de-novo-validacao-cruzada-aninhada), para que ele avalie os passos 7 a 9 juntos: as duas buscas *e* a escolha entre elas. O número cai mais?
4. **Mais candidatos.** Acrescente à tabela do passo 8 um candidato SMOTE + k-NN (um pipeline do `imblearn`, como na [aula de ROC e dados desbalanceados](https://insper.github.io/ML/pt/2026.2/classes/roc-imbalanced/)) ajustado no mesmo `cv` e compare-o com o vencedor fold a fold. Depois da [aula de SVM](https://insper.github.io/ML/pt/2026.2/classes/svm/), volte e acrescente também um `SVC`. Algum deles vence a regressão logística ajustada por mais que a dispersão das diferenças pareadas?

### Sua vez (fundamental) — exercício 1: quantos folds?

In [18]:
import time

metrics = ['accuracy', 'precision', 'recall', 'f1', 'roc_auc']
rows = []
for k in [3, 5, 10]:
    cv_k = StratifiedKFold(n_splits=k, shuffle=True, random_state=42)
    t0 = time.perf_counter()
    r = cross_validate(model, X_train, y_train, cv=cv_k, scoring=metrics)
    rows.append({'folds': k, 'recall': r['test_recall'].mean(), 'recall std': r['test_recall'].std(),
                 'ROC AUC': r['test_roc_auc'].mean(), 'AUC std': r['test_roc_auc'].std(),
                 'fit time total (s)': r['fit_time'].sum(), 'wall (s)': time.perf_counter() - t0})
print(pd.DataFrame(rows).round(3).to_string(index=False))

 folds  recall  recall std  ROC AUC  AUC std  fit time total (s)  wall (s)
     3   0.959       0.008    0.996    0.004               0.013     0.034
     5   0.953       0.040    0.996    0.005               0.020     0.054
    10   0.959       0.046    0.995    0.008               0.044     0.109


**Conclusão.** A média quase não muda: recall 0,959, 0,953 e 0,959 com 3, 5 e 10 folds. O desvio entre folds cresce com $k$ (0,008, 0,040 e 0,046), porque cada fold de validação fica menor (com 10 folds, cerca de 17 malignos, e um tumor vale quase 6 pontos); o 0,008 de 3 folds é também sorte de ter só 3 valores. O custo de ajuste cresce proporcional a $k$ (0,017 s, 0,029 s e 0,060 s no total), irrelevante aqui. (O desvio desta tabela é o populacional do NumPy; o 0,045 do passo 4 é o do pandas, com `ddof=1`.)

### Sua vez (fundamental) — exercício 2: sem estratificar

In [19]:
from sklearn.model_selection import KFold


def positives_per_fold(splitter, X_, y_):
    return [int(y_.iloc[va].sum()) for _, va in splitter.split(X_, y_)]


kf = KFold(n_splits=5, shuffle=True, random_state=42)
print(f"37% malignos · KFold {positives_per_fold(kf, X_train, y_train)} · "
      f"StratifiedKFold {positives_per_fold(cv, X_train, y_train)}")

# o mesmo treino com só 3% de malignos: todos os benignos e uma amostra de malignos
benign = y_train[y_train == 0].index
n_mal = round(0.03 * len(benign) / 0.97)
malign = y_train[y_train == 1].sample(n_mal, random_state=42).index
X3, y3 = X_train.loc[benign.union(malign)], y_train.loc[benign.union(malign)]
print(f"3% malignos ({y3.sum()} de {len(y3)}) · KFold {positives_per_fold(kf, X3, y3)} · "
      f"StratifiedKFold {positives_per_fold(cv, X3, y3)}")
for seed in range(20):                       # procurando uma semente que deixa um fold sem malignos
    counts = positives_per_fold(KFold(n_splits=5, shuffle=True, random_state=seed), X3, y3)
    if 0 in counts:
        print(f"KFold com random_state={seed}: {counts}")
        break

37% malignos · KFold [32, 39, 30, 38, 31] · StratifiedKFold [34, 34, 34, 34, 34]
3% malignos (9 de 294) · KFold [0, 2, 1, 5, 1] · StratifiedKFold [2, 2, 2, 2, 1]
KFold com random_state=1: [3, 0, 0, 6, 0]


**Conclusão.** Com 37% de malignos, o `KFold` põe de 30 a 39 malignos por fold de validação contra 34 em todos com estratificação: diferença pequena. Com 3% (9 malignos em 294 linhas), o `KFold` com a mesma semente já deixa um fold com **zero** malignos ([0, 2, 1, 5, 1]) e, com outra semente, três ([3, 0, 0, 6, 0]); nesses folds o recall nem está definido e um fold concentra metade dos positivos. A estratificação ([2, 2, 2, 2, 1]) vira obrigatória porque garante positivos em todo fold de validação.

### Sua vez (fundamental) — exercício 3: o scaler fora do pipeline

In [20]:
X_train_scaled = StandardScaler().fit_transform(X_train)        # vazamento: o scaler viu os folds de validação
lr = LogisticRegression(max_iter=1000)
for name, est, X_ in [('scaler no pipeline (passo 4)', model, X_train),
                      ('scaler fora do pipeline', lr, X_train_scaled)]:
    r = cross_validate(est, X_, y_train, cv=cv, scoring=['accuracy', 'recall', 'roc_auc'])
    print(f"{name:28s} acurácia {r['test_accuracy'].mean():.4f} · recall {r['test_recall'].mean():.4f} · "
          f"ROC AUC {r['test_roc_auc'].mean():.4f}")

scaler no pipeline (passo 4) acurácia 0.9736 · recall 0.9529 · ROC AUC 0.9958
scaler fora do pipeline      acurácia 0.9736 · recall 0.9529 · ROC AUC 0.9958


**Conclusão.** Aqui a diferença é nula até a quarta casa: acurácia 0,9736, recall 0,9529 e ROC AUC 0,9958 nos dois casos, porque a média e o desvio estimados em 455 linhas mal mudam em relação aos estimados em 364. Isso não prova que é seguro: o protocolo continua vazando a validação para o ajuste, e com etapas que aprendem mais dos dados (seleção de atributos usando `y`, SMOTE, codificação pelo alvo) ou com poucas linhas o mesmo erro infla muito a estimativa.

### Sua vez (avançado) — exercício 1: limiar

In [21]:
from sklearn.metrics import precision_recall_curve, precision_score

proba_oof = cross_val_predict(best, X_train, y_train, cv=cv, method='predict_proba')[:, 1]
prec, rec_curve, thr = precision_recall_curve(y_train, proba_oof)
ok = np.where(rec_curve[:-1] >= 0.98)[0]
i = ok[np.argmax(thr[ok])]                  # maior limiar com recall >= 0,98
t = thr[i]
default = (proba_oof >= 0.5).astype(int)
print(f"fora do fold, limiar 0,5:  recall {recall_score(y_train, default):.3f} · "
      f"precisão {precision_score(y_train, default):.3f}")
print(f"fora do fold, limiar {t:.3f}: recall {rec_curve[i]:.3f} · precisão {prec[i]:.3f}")

pred_t = (proba >= t).astype(int)            # aplicado uma vez ao teste
print(f"teste, limiar {t:.3f}: recall {recall_score(y_test, pred_t):.3f} · "
      f"precisão {precision_score(y_test, pred_t):.3f}")
print(confusion_matrix(y_test, pred_t, labels=[1, 0]))
print(f"teste, limiar 0,5 (passo 12): recall {recall_score(y_test, pred):.3f} · "
      f"precisão {precision_score(y_test, pred):.3f}")

fora do fold, limiar 0,5:  recall 0.959 · precisão 0.976
fora do fold, limiar 0.314: recall 0.982 · precisão 0.928
teste, limiar 0.314: recall 0.976 · precisão 0.911
[[41  1]
 [ 4 68]]
teste, limiar 0,5 (passo 12): recall 0.976 · precisão 0.976


**Conclusão.** Fora do fold, o maior limiar com recall ≥ 0,98 é 0,314: o recall vai de 0,959 para 0,982 e a precisão cai de 0,976 para 0,928, ou seja, cerca de 5 pontos de precisão pagos por 2,3 de recall. Aplicado uma vez ao teste, o limiar não pegou o tumor que faltava (recall 0,976, os mesmos 41 de 42) e trouxe 3 falsos alarmes a mais (precisão 0,911 contra 0,976). Com 42 malignos, um tumor é 2,4 pontos, então o teste é ruidoso demais para julgar o limiar; ele foi escolhido no treino e o resultado é o que é.

### Sua vez (avançado) — exercício 2: penalidade L1

In [22]:
import warnings
from sklearn.exceptions import ConvergenceWarning

warnings.filterwarnings('ignore', category=ConvergenceWarning)   # liblinear não converge com C grande; irrelevante perto do melhor C
model_l1 = make_pipeline(StandardScaler(), LogisticRegression(l1_ratio=1, solver='liblinear', max_iter=1000))
grid_l1 = GridSearchCV(model_l1, {'logisticregression__C': np.logspace(-3, 3, 13)},
                       cv=cv, scoring='recall', return_train_score=True)
grid_l1.fit(X_train, y_train)

table_l1 = pd.DataFrame(grid_l1.cv_results_)[['param_logisticregression__C', 'mean_train_score',
                                              'mean_test_score', 'std_test_score']]
table_l1.columns = ['C', 'recall treino', 'recall val', 'desvio val']
print(table_l1.round(3).to_string(index=False))
coef_l1 = grid_l1.best_estimator_[-1].coef_[0]
std_l1 = grid_l1.cv_results_['std_test_score'][grid_l1.best_index_]
std_l2 = grid.cv_results_['std_test_score'][grid.best_index_]
print(f"L1 melhor: C = {grid_l1.best_params_['logisticregression__C']:.3g} · recall {grid_l1.best_score_:.3f} ± {std_l1:.3f} · "
      f"{(coef_l1 == 0).sum()} de {len(coef_l1)} coeficientes exatamente zero")
print(f"L2 (passo 6): C = {grid.best_params_['logisticregression__C']:g} · recall {grid.best_score_:.3f} ± {std_l2:.3f}")
print("atributos que sobram:", list(X.columns[coef_l1 != 0]))

       C  recall treino  recall val  desvio val
   0.001          0.000       0.000       0.000
   0.003          0.000       0.000       0.000
   0.010          0.954       0.959       0.030
   0.032          0.960       0.959       0.030
   0.100          0.962       0.953       0.030
   0.316          0.962       0.941       0.042
   1.000          0.976       0.947       0.034
   3.162          0.979       0.947       0.034
  10.000          0.988       0.941       0.032
  31.623          1.000       0.941       0.032
 100.000          1.000       0.941       0.032
 316.228          1.000       0.947       0.022
1000.000          1.000       0.947       0.022
L1 melhor: C = 0.01 · recall 0.959 ± 0.030 · 26 de 30 coeficientes exatamente zero
L2 (passo 6): C = 1 · recall 0.953 ± 0.040
atributos que sobram: ['mean concave points', 'worst radius', 'worst perimeter', 'worst concave points']


**Conclusão.** No melhor `C` (0,01) a L1 zera 26 dos 30 coeficientes e mantém só quatro atributos (`mean concave points`, `worst radius`, `worst perimeter` e `worst concave points`). O recall é 0,959 ± 0,030, contra 0,953 ± 0,040 da L2 no passo 6: uma diferença de 0,006, muito menor que um desvio. Com `C` ≤ 0,003 todos os coeficientes zeram e o recall vai a 0; nos `C` grandes o liblinear não converge em 1000 iterações (aviso silenciado), longe do melhor `C`.

### Sua vez (avançado) — exercício 3: a escolha inteira, aninhada

In [23]:
outer_scores, winners = [], []
for tr, va in outer.split(X_train, y_train):      # o mesmo cv externo da simulação do passo 12
    Xo_tr, yo_tr = X_train.iloc[tr], y_train.iloc[tr]
    fitted = {name: clone(s).fit(Xo_tr, yo_tr) for name, s in [('logística', search_lr), ('k-NN', search_knn)]}
    name = max(fitted, key=lambda n: fitted[n].best_score_)   # passo 9: escolhe pelo recall interno
    winners.append(name)
    outer_scores.append(recall_score(y_train.iloc[va], fitted[name].predict(X_train.iloc[va])))
outer_scores = np.array(outer_scores)
print("vencedor em cada fold externo:", winners)
print(f"escolha inteira aninhada: {outer_scores.mean():.3f} ± {outer_scores.std():.3f} · folds {outer_scores.round(3)}")
print(f"só a busca da logística aninhada: {nested.mean():.3f} · best_score_: {search_lr.best_score_:.3f}")

vencedor em cada fold externo: ['logística', 'logística', 'logística', 'logística', 'logística']
escolha inteira aninhada: 0.947 ± 0.022 · folds [0.912 0.971 0.941 0.941 0.971]
só a busca da logística aninhada: 0.947 · best_score_: 0.959


**Conclusão.** A logística vence a escolha interna nos 5 folds externos, então a avaliação da escolha inteira dá 0,947 ± 0,022, exatamente o mesmo número da CV aninhada só da logística. O número não cai mais porque a escolha entre as famílias é estável; ainda está 1,2 ponto abaixo do `best_score_` (0,959), dentro do ruído.

### Sua vez (avançado) — exercício 4: mais candidatos

In [24]:
from imblearn.over_sampling import SMOTE
from imblearn.pipeline import make_pipeline as make_imb_pipeline
from sklearn.svm import SVC

search_smote = GridSearchCV(
    make_imb_pipeline(StandardScaler(), SMOTE(random_state=42), KNeighborsClassifier()),
    {'kneighborsclassifier__n_neighbors': [1, 3, 5, 7, 9, 11, 15, 21, 31],
     'kneighborsclassifier__weights': ['uniform', 'distance']},
    cv=cv, scoring=['recall', 'roc_auc'], refit='recall')
search_svc = GridSearchCV(
    make_pipeline(StandardScaler(), SVC()),
    {'svc__C': [0.1, 1, 10, 100], 'svc__gamma': [0.001, 0.01, 0.1], 'svc__class_weight': [None, 'balanced']},
    cv=cv, scoring=['recall', 'roc_auc'], refit='recall')
for name, search in [('SMOTE + k-NN', search_smote), ('SVC', search_svc)]:
    search.fit(X_train, y_train)
    print(f"{name}: {search.best_params_}")
    scores[name + ', tuned'] = cross_validate(search.best_estimator_, X_train, y_train, cv=cv,
                                              scoring=['recall', 'roc_auc'])

summary = pd.DataFrame({name: {'recall': s['test_recall'].mean(), 'recall std': s['test_recall'].std(),
                               'ROC AUC': s['test_roc_auc'].mean(), 'AUC std': s['test_roc_auc'].std()}
                        for name, s in scores.items()}).T
print(summary.round(3))

ref = scores['logistic, tuned']
for name in ['SMOTE + k-NN, tuned', 'SVC, tuned']:
    for metric in ['recall', 'roc_auc']:
        d = scores[name]['test_' + metric] - ref['test_' + metric]
        print(f"{name} − logística ({metric:7s}): por fold {d.round(3)} · média {d.mean():+.3f} · desvio {d.std():.3f}")

SMOTE + k-NN: {'kneighborsclassifier__n_neighbors': 5, 'kneighborsclassifier__weights': 'uniform'}


SVC: {'svc__C': 0.1, 'svc__class_weight': 'balanced', 'svc__gamma': 0.1}
                     recall  recall std  ROC AUC  AUC std
dummy                 0.000       0.000    0.500    0.000
logistic, default     0.953       0.040    0.996    0.005
logistic, tuned       0.959       0.030    0.995    0.005
k-NN, default         0.918       0.034    0.987    0.014
k-NN, tuned           0.935       0.039    0.989    0.013
SMOTE + k-NN, tuned   0.965       0.022    0.988    0.013
SVC, tuned            0.959       0.035    0.982    0.008
SMOTE + k-NN, tuned − logística (recall ): por fold [ 0.059  0.029 -0.059  0.     0.   ] · média +0.006 · desvio 0.039
SMOTE + k-NN, tuned − logística (roc_auc): por fold [ 0.001  0.001 -0.005 -0.02  -0.013] · média -0.007 · desvio 0.008
SVC, tuned − logística (recall ): por fold [ 0.     0.029  0.     0.    -0.029] · média +0.000 · desvio 0.019
SVC, tuned − logística (roc_auc): por fold [-0.018 -0.002 -0.018 -0.009 -0.019] · média -0.013 · desvio 0.007


**Conclusão.** O SMOTE + k-NN ($k = 5$) faz recall 0,965 ± 0,022 e o `SVC` (`C = 0,1`, `gamma = 0,1`, `balanced`) 0,959 ± 0,035, contra 0,959 ± 0,030 da logística ajustada. Fold a fold, nenhum vence por mais que a dispersão: SMOTE + k-NN − logística = +0,006 ± 0,039 (+2, +1, −2, 0 e 0 tumores) e `SVC` − logística = +0,000 ± 0,019. Na ROC AUC os dois perdem para a logística (−0,007 ± 0,008 e −0,013 ± 0,007, este em todos os 5 folds), então a logística ajustada continua sendo a escolha.